# Performance Comparison Lab

```{contents}
:local:
:depth: 2
```

## Match the Structure to the Workload

```{index} workload; data-structure choice
```

The same service desk has several access needs. Use a priority queue for the next urgent ticket, a dictionary for its owner, and a set for uniqueness or closed-ticket membership. A tree supports ordered keys and range queries. Each structure preserves a different relationship; the application must keep shared IDs consistent.

| Need | Structure | Cost under stated assumptions |
| --- | --- | --- |
| Select next urgency | Heap priority queue | O(log n) repair; backing-array growth can make one enqueue O(n). |
| Find owner by ID | Hash dictionary | Expected O(1) with well-distributed hashing and bounded load; worst-case O(n). |
| Reject duplicate ID | Hash set or dictionary | Expected O(1) under the same assumptions. |
| Ordered interval report | Balanced search tree | O(log n + k) for k results; the Chapter 17 unbalanced tree depends on actual height. |
| Process by arrival | FIFO queue | Constant-time endpoints in the linked representation. |

These bounds assume fixed-cost comparison and hashing. They describe operations, not measured milliseconds. A tiny list can still be a practical choice when there are few records and little repeated lookup.

Download {download}`the complete heap and chained-map source with verification <../../materials/18/HeapHashStructures.cs>` to use in a console project.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var urgentWork = new PriorityQueue<string, int>();
urgentWork.Enqueue("Weekly report", 5); urgentWork.Enqueue("Database outage", 0);
var ownerByTicket = new Dictionary<string, string> { ["T-100"] = "Amina" };
var closedTickets = new HashSet<string> { "T-099" };
Console.WriteLine($"next={urgentWork.Dequeue()}");
Console.WriteLine($"owner={ownerByTicket["T-100"]}");
Console.WriteLine($"closed={closedTickets.Contains("T-099")}");

## Count Work Before Timing

```{index} hashing; operation count; performance measurement
```

One stopwatch measurement includes runtime warm-up, scheduling noise, and timer granularity. It cannot establish a complexity bound. First compare the amount of relevant work. In the controlled example below, the list must inspect every key for an absent lookup, while an explicitly constructed chain table inspects only the selected bucket.

The small hash function deliberately maps nonnegative integer IDs by remainder. This is a reproducible placement experiment, not instrumentation of .NET Dictionary.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

int[] ids = Enumerable.Range(0, 1000).ToArray();
int target = 1999;
int linearComparisons = 0;
foreach (int id in ids) { linearComparisons++; if (id == target) break; }
var buckets = new List<int>[100];
for (int i = 0; i < buckets.Length; i++) buckets[i] = new List<int>();
foreach (int id in ids) buckets[id % buckets.Length].Add(id);
int bucketComparisons = 0;
foreach (int id in buckets[target % buckets.Length])
{ bucketComparisons++; if (id == target) break; }
Console.WriteLine($"linear comparisons={linearComparisons}");
Console.WriteLine($"selected-chain comparisons={bucketComparisons}");

The absent lookup uses 1000 comparisons in the list and 10 in the selected chain. With a constant hash function, all keys would occupy one chain and the lookup would again require 1000 comparisons. Collision distribution explains the difference.

A benchmark should separate construction from the operation being studied, repeat lookups over representative hit/miss keys, warm up the runtime, report variation, and include realistic key-hashing costs. Those measurements complement the analysis; they do not turn expected constant-time hashing into a worst-case guarantee. Chapter 20 develops analysis and measurement further.

## Top-K with a Min-Heap

```{index} top-k; min-heap; streaming
```

Keep the k largest observed transaction amounts without sorting the entire input. Maintain at most k retained values in a min-heap. The root is the smallest retained value, so it is the candidate to discard when a larger value arrives.

**Invariant:** after each input, the heap contains the largest `min(k, valuesSeen)` values of that prefix, including repeated values. If fewer than k values are stored, add the next value. Once full, a value no larger than the root cannot improve the retained multiset; a larger one replaces that root.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

int[] amounts = { 72, 91, 84, 99, 67, 88, 95, 95 };
int k = 3;
Console.WriteLine("top=" + string.Join(", ", TopK(amounts, k)));
Console.WriteLine($"zero-k count={TopK(amounts, 0).Count}");

List<int> TopK(IEnumerable<int> values, int k)
{
    if (k < 0) throw new ArgumentOutOfRangeException(nameof(k));
    if (k == 0) return new List<int>();
    var retained = new PriorityQueue<int, int>();
    foreach (int value in values)
    {
        if (retained.Count < k) retained.Enqueue(value, value);
        else if (value > retained.Peek())
        {
            retained.Dequeue();
            retained.Enqueue(value, value);
        }
    }
    var result = new List<int>();
    while (retained.TryDequeue(out int value, out _)) result.Add(value);
    result.Reverse();
    return result;
}

For k = 3, retained values after the first three inputs are 72, 84, 91. Processing 99 drops 72; 67 changes nothing; 88 drops 84; 95 drops 88; the second 95 drops 91. The final result is `99, 95, 95`. Repeated equal amounts remain separate observations.

For n inputs and positive k, maintenance has an O(n log(k + 1)) upper bound and O(k) storage, or fewer entries when n < k. Producing the final ordered result costs O(k log(k + 1)) in the worst case. k = 0 returns immediately; negative k is rejected. If record identity or tie order matters, include an explicit secondary priority and define which tied records to retain.

## Dispatch from Two Consistent Indexes

```{index} priority queue; dictionary; consistency
```

The service desk keeps active records in a dictionary and pending IDs in a priority queue. A simple cancellation policy removes the active record while leaving its old queue entry in place. On dequeue, skip IDs no longer active. This is **lazy deletion**: defer physical cleanup until an obsolete entry reaches the front.

This example allows each ID to be registered once during the batch. If an ID can be reused or reprioritized, an old entry may still refer to an active ID. Use a generation/version field and verify it on dequeue, or implement an indexed heap with explicit updates. Presence alone would then be insufficient.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var active = new Dictionary<string, string>();
var registered = new HashSet<string>();
var pending = new PriorityQueue<string, (int Urgency, long Arrival)>();
long arrival = 0;
bool Register(string id, string owner, int urgency)
{
    if (!registered.Add(id)) return false;
    active.Add(id, owner);
    pending.Enqueue(id, (urgency, arrival++));
    return true;
}
Register("T-100", "Amina", 2);
Register("T-101", "Diego", 0);
Register("T-102", "Mina", 2);
Console.WriteLine($"duplicate accepted={Register("T-100", "Other", 0)}");
active.Remove("T-101");
while (pending.TryDequeue(out string? id, out _))
{
    if (!active.TryGetValue(id, out string? owner)) { Console.WriteLine($"skip cancelled {id}"); continue; }
    active.Remove(id);
    Console.WriteLine($"serve {id}: {owner}");
}
Console.WriteLine($"active={active.Count}, pending={pending.Count}");

The cancelled urgent ticket is skipped, then T-100 and T-102 are served in their arrival order. Immediately after cancellation, the dictionary has two entries but the queue still has three. Equal Counts are not the invariant. Every active record has one live pending entry, while pending may also contain inactive IDs.

Skipping one stale entry costs one heap removal. Across a batch, each enqueued entry is removed at most once, but a single request for the next active ticket may skip many cancelled entries. Cancellation also leaves queue storage occupied until cleanup. Analyze both the whole workload and an individual call; do not promise that lazy deletion makes every dispatch O(log n).

## Exercise

```{index} top-k; exercise
```

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Exercise: keep the two largest transaction amounts.
int[] amounts = { 70, 95, 82, 99, 88 };
var top = new PriorityQueue<int, int>();
// Your code starts here.
/* TODO */
// Your code ends here.
var result = new List<int>();
while (top.TryDequeue(out int amount, out _)) result.Add(amount);
result.Reverse();
Console.WriteLine(string.Join(", ", result));

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

// Solution
int[] amounts = { 70, 95, 82, 99, 88 };
var top = new PriorityQueue<int, int>();
foreach (int amount in amounts)
{
    top.Enqueue(amount, amount);
    if (top.Count > 2) top.Dequeue();
}
var result = new List<int>();
while (top.TryDequeue(out int amount, out _)) result.Add(amount);
result.Reverse();
Console.WriteLine(string.Join(", ", result));

99, 95


## Summary

```{index} hashing; heaps; workload analysis
```

Use heaps for priority selection and hash collections for equality-based access. State tie and duplicate policies, test boundaries, and distinguish average or amortized behavior from a costly individual operation. When two indexes describe the same business records, define which entries are live and how obsolete entries are recognized.

```{rubric} Footnotes
```

[^1]: Lazy deletion trades simpler cancellation for additional pending entries and occasional cleanup bursts. Production schedulers may need versioned entries, bounded cleanup, or an indexed heap.